In [ ]:
from google.genai import types
import json
import chromadb
import sys
from SharedResources.load_environment import get_client
from chromadb import PersistentClient
client = get_client()
def get_embeddings(table_data:dict,model_id: str):
    global client
    combined_text = f"""Table Name : {table_data.get('table_name')}
Description : {table_data.get('description')}
"""

    response = client.models.embed_content(
        model=model_id,
        contents=combined_text,
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY",
                                        # output_dimensionality=768
                                        ),
    )
    emb = response.embeddings[0]
    values = getattr(emb, "values", emb)
    return list(values)





environment varialble loaded!
Client created successfully!


In [2]:

# uncomment for adding documents into collections
def add_document(meta_data_path : str , model_id: str = "text-embedding-005"):
    
    with open(meta_data_path,'r') as f:
        table_metadata = json.load(f)
    try:
        for table_data in table_metadata:
            embedding = get_embeddings(table_data,model_id)
            collection.add(ids = [table_data['table_name']],
                           documents =
                               f"""Description : {table_data['table_description']}
                               Schema : {table_data['create_statement']}""",
                          embeddings = [embedding],
                          metadatas=[
                              {"create_statement" : table_data['create_statement'],
                              "columns": str(table_data['columns'])
                               }]
                           # ----- METADATA CANNOT BE OF DICTIONARY TYPE
                           )
    except Exception as e:
        return f"There was a error loading the documents. \n{e}"
    

    
chroma_client = chromadb.PersistentClient(path = "./chromadb")
collection = chroma_client.get_or_create_collection(name = "SQL_MetaStore")
add_document('./table_metadata.json')

In [3]:
#test embedding
collection = chroma_client.get_collection(name="SQL_MetaStore") # getting the vector collection

response = client.models.embed_content(
model="text-embedding-005",
contents="Customer details with order status as cancelled.",
config=types.EmbedContentConfig(
    task_type="RETRIEVAL_DOCUMENT"
),).embeddings[0].values

# embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")
similar_tables = collection.query(query_embeddings = response, n_results = 3)

In [4]:
from pprint import pp
pp(similar_tables)

{'ids': [['customers', 'orders', 'order_items']],
 'embeddings': None,
 'documents': [['Description : Contains customer (user, client, account '
                'holder) profile details such as name, email, phone, and '
                'geographic information. Used to answer queries related to '
                'customer details, user profiles, premium users, and customer '
                'demographics. Each customer is uniquely identified by '
                'customer_id. This table is commonly joined with orders, '
                'subscriptions, support_tickets, and reviews using '
                'customer_id. Frequently filtered using country, city, '
                'created_at, and is_premium status.\n'
                '                               Schema : CREATE TABLE '
                'customers (customer_id STRING, name STRING, email STRING, '
                'phone STRING, created_at TIMESTAMP, country STRING, city '
                'STRING, is_premium BOOL);',
        